
## Notebook 02: Daily Full Moments Factor Inventory
**Input:** `Data/Data_Collection/Final/Stage_3_Model_Ready/model_market_daily_full_moments.parquet`
**Inventories:** `stock_daily_factor_inventory_final.csv`, `macro_daily_factor_inventory_final.csv`

### Logic
Same schema-read approach as Notebook 01. Features fall into two categories:

- **Stock moment columns** (named `{base_factor}_{moment_type}`): the suffix is stripped to recover the base factor name, which is looked up in the stock inventory. The five recognised suffixes are `_cwmean`, `_cwstd`, `_cwskew`, `_cwkurt`, `_spread`. The description is prefixed with the moment type (e.g., "cwstd of: Daily share turnover..."). The same `skew_chg_5d` → `stock_skew_chg_5d` rename is handled.
- **Macro columns** (no moment suffix): looked up directly in the macro inventory as raw levels.

Unmatched columns are flagged. A summary of feature counts by moment type is printed.

### Output Columns
`column`, `base_factor`, `moment_type` (cwmean / cwstd / cwskew / cwkurt / spread / raw level), `panel`, `source`, `category`, `description`

**Output:** `Data/Data_Collection/Final/Stage_3_Model_Ready/daily_full_moments_factor_inventory.csv`

---

## Key Notes
- Neither notebook loads the full parquet data -- only the schema is read via `pyarrow.parquet.read_schema`, making both notebooks nearly instantaneous.
- The `stock_skew_chg_5d` rename is the only naming conflict in the daily tables. It arose because `skew_chg_5d` existed in both Panel A (stock-level OTM skew change) and Panel C (CBOE SKEW 5-day change), and was resolved with a `stock_` prefix during the Stage 2 merge.
- Unmatched columns would indicate a gap between the Stage 1.5 inventories and the actual features in the model-ready tables, which would require investigation.

In [1]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

# Load feature names from daily full moments
BASE = Path('../../../../Data/Data_Collection/Final/Stage_3_Model_Ready')
schema = pq.read_schema(BASE / 'model_market_daily_full_moments.parquet')
all_cols = [f.name for f in schema]
features = [c for c in all_cols if c not in ['date', 'target_daily_return']]

# Load both inventories
INV_DIR = Path('../../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering')
stock_inv = pd.read_csv(INV_DIR / 'stock_daily_factor_inventory_final.csv')
macro_inv = pd.read_csv(INV_DIR / 'macro_daily_factor_inventory_final.csv')

stock_inv_map = stock_inv.set_index('column')[['source', 'category', 'description']].to_dict('index')
macro_inv_map = macro_inv.set_index('column')[['source', 'category', 'description']].to_dict('index')

# Handle the renamed conflict
stock_inv_map['stock_skew_chg_5d'] = stock_inv_map.get('skew_chg_5d', {})

# Moment suffixes to strip for base factor lookup
moment_suffixes = ['_cwmean', '_cwstd', '_cwskew', '_cwkurt', '_spread']

rows = []
matched = 0
unmatched = []

for f in features:
    # Check if it's a moment column (stock factor with suffix)
    is_moment = False
    for suffix in moment_suffixes:
        if f.endswith(suffix):
            base_name = f[:-len(suffix)]
            moment_type = suffix[1:]  # remove leading underscore
            
            # Handle renamed conflict
            if base_name == 'stock_skew_chg_5d':
                lookup = 'skew_chg_5d'
            else:
                lookup = base_name
            
            if lookup in stock_inv_map:
                info = stock_inv_map[lookup]
                rows.append({
                    'column': f,
                    'base_factor': base_name,
                    'moment_type': moment_type,
                    'panel': 'A (stock daily)',
                    'source': info.get('source', ''),
                    'category': info.get('category', ''),
                    'description': f"{moment_type} of: {info.get('description', '')}",
                })
                matched += 1
                is_moment = True
                break
    
    if is_moment:
        continue
    
    # Check macro inventory (raw levels, no suffix)
    if f in macro_inv_map:
        info = macro_inv_map[f]
        rows.append({
            'column': f,
            'base_factor': f,
            'moment_type': 'raw level',
            'panel': 'C (macro daily)',
            'source': info.get('source', ''),
            'category': info.get('category', ''),
            'description': info.get('description', ''),
        })
        matched += 1
    else:
        rows.append({
            'column': f,
            'base_factor': f,
            'moment_type': '???',
            'panel': '???',
            'source': '',
            'category': '',
            'description': '',
        })
        unmatched.append(f)

result = pd.DataFrame(rows)

print(f"Total features: {len(features)}")
print(f"Matched: {matched}")
print(f"Unmatched: {len(unmatched)}")
if unmatched:
    print(f"\nUnmatched columns:")
    for c in unmatched:
        print(f"  {c}")

# Summary by moment type
print(f"\nBy moment type:")
print(result['moment_type'].value_counts().to_string())

# Save
out_path = Path('../../../../Data/Data_Collection/Final/Stage_3_Model_Ready/daily_full_moments_factor_inventory.csv')
result.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")
print(f"  {len(result)} rows")

Total features: 1145
Matched: 1145
Unmatched: 0

By moment type:
moment_type
raw level    209
cwmean       189
cwstd        189
spread       188
cwskew       185
cwkurt       185

Saved: ..\..\..\..\Data\Data_Collection\Final\Stage_3_Model_Ready\daily_full_moments_factor_inventory.csv
  1145 rows
